# 06 — Finding features from errors

The feature loop, starting at exp_0005 as promised: **segment the OOF errors →
hypothesise → one experiment → verdict.** Features come from where the model fails, not
from imagination — plus the one theory-driven source, arithmetic a tree cannot build
(LEARNING.md: ratios, weak-pair interactions, cross-row aggregations).

The first two experiments came back negative, and section 4 is about *why the loop
produced only two ideas*. Its blind spot has a name — it asks **where** the model fails
and never **at what resolution** — and closing it turns up the one feature idea in this
competition that measurably pays.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import polars as pl

from s6e7 import cv, eda, io

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_width_chars(200)
train = io.load_train()
oof_base = np.load(cv.OOF_DIR / "exp_0001.npy")

---
## 1. Where does the baseline fail?

`eda.recall_by_bin` slices the OOF errors along one feature at a time. The first row is
the global reference; a bin whose recall craters below it marks a segment the current
features cannot express. `sleep_duration` — the strongest single feature — is the
standout:

In [2]:
eda.recall_by_bin(train, oof_base, "sleep_duration", io.TARGET, n_bins=8).select(
    "bin", "n_rows", "recall_at-risk", "recall_fit", "recall_unhealthy", "balanced_acc"
)

bin,n_rows,recall_at-risk,recall_fit,recall_unhealthy,balanced_acc
str,i64,f64,f64,f64,f64
"""all""",690088,0.9927,0.8254,0.8007,0.8729
"""3 - 5.63""",76054,0.9876,0.0488,0.9083,0.6482
"""5.63 - 6.16""",77375,0.9858,0.0188,0.8681,0.6242
"""6.16 - 6.62""",74340,1.0,0.0,0.0,0.3333
"""6.62 - 6.99""",79158,0.9999,0.009,0.0,0.3363
"""6.99 - 7.37""",75669,0.9964,0.8613,0.0,0.6193
"""7.37 - 7.81""",76910,0.997,0.8711,0.0,0.6227
"""7.81 - 8.39""",77616,0.997,0.871,0.0058,0.6246
"""8.39 - 10""",76967,0.9962,0.8747,0.005,0.6253


Three readings:

- **The minorities live at the extremes.** `fit` recall is ~0 below 6.2 hours and ~0.87
  above 7; `unhealthy` is the mirror image. In the middle band (6.2–7.0) *both* minority
  recalls are ~0 — where sleep is uninformative, the other 12 features barely separate
  the classes at all.
- **The null bin is a crater.** When `sleep_duration` is missing, `unhealthy` recall
  falls 0.80 → 0.21. The model leans on sleep so hard that its absence takes the
  minorities down with it.
- `at-risk` never suffers — the majority class is the default answer everywhere.

The activity pair shows the same pattern for `fit`:

In [3]:
for col in ("step_count", "exercise_duration"):
    t = eda.recall_by_bin(train, oof_base, col, io.TARGET, n_bins=8)
    print(t.select("bin", "n_rows", "recall_fit", "balanced_acc").head(4))

shape: (4, 4)
┌─────────────┬────────┬────────────┬──────────────┐
│ bin         ┆ n_rows ┆ recall_fit ┆ balanced_acc │
│ ---         ┆ ---    ┆ ---        ┆ ---          │
│ str         ┆ i64    ┆ f64        ┆ f64          │
╞═════════════╪════════╪════════════╪══════════════╡
│ all         ┆ 690088 ┆ 0.8254     ┆ 0.8729       │
│ 1002 - 3378 ┆ 84407  ┆ 0.02       ┆ 0.6049       │
│ 3378 - 5389 ┆ 84396  ┆ 0.1661     ┆ 0.6518       │
│ 5389 - 7150 ┆ 84760  ┆ 0.7511     ┆ 0.8504       │
└─────────────┴────────┴────────────┴──────────────┘
shape: (4, 4)
┌─────────────┬────────┬────────────┬──────────────┐
│ bin         ┆ n_rows ┆ recall_fit ┆ balanced_acc │
│ ---         ┆ ---    ┆ ---        ┆ ---          │
│ str         ┆ i64    ┆ f64        ┆ f64          │
╞═════════════╪════════╪════════════╪══════════════╡
│ all         ┆ 690088 ┆ 0.8254     ┆ 0.8729       │
│ 0 - 22.2    ┆ 84500  ┆ 0.0052     ┆ 0.5988       │
│ 22.2 - 29.2 ┆ 86244  ┆ 0.1608     ┆ 0.6517       │
│ 29.2 - 34.5 ┆ 84

---
## 2. Two hypotheses, one experiment each

**H1 — explicit missingness indicators (exp_0005).** The null-bin craters plus the known
`bmi_is_null` signal (unhealthy 2.79% vs 8.47%, ~22σ) suggest handing the model 13
explicit is-null flags. Counter-argument, known in advance: LightGBM already routes NaN
at every split, so the flags may be redundant.

**H2 — activity-intensity ratios (exp_0006).** `fit` recall dies at low steps and low
exercise minutes, and the activity trio is internally correlated (0.37–0.44). A *ratio*
(calories per step, per exercise-minute) is exactly the diagonal combination a tree
cannot build from axis-aligned splits — if intense-but-brief exercisers are hiding in the
low bins, ratios expose them.

In [4]:
from s6e7.cv import ExperimentConfig

for exp_id, feats, changed in [
    ("exp_0005", "indicators", "add 13 missingness indicator columns"),
    ("exp_0006", "ratios", "add 3 activity-intensity ratios (from exp_0001 error profile)"),
]:
    r = cv.run(ExperimentConfig(exp_id, "lgbm", features=feats, parent="exp_0001", changed=changed),
               train=train, test=io.load_test(), if_logged="skip")
    print(f"{exp_id}: cv {r.cv_mean:.5f} +/- {r.cv_std:.5f}")

exp_0005: cv 0.87291 +/- 0.00205
exp_0006: cv 0.87285 +/- 0.00200


## 3. Verdicts — paired, as always

In [5]:
cv.paired_diff("exp_0005", "exp_0001", train=train)

fold,exp_0005,exp_0001,diff,t
str,f64,f64,f64,null
"""0""",0.87121,0.87121,0.0,null
"""1""",0.87515,0.87515,0.0,null
"""2""",0.87502,0.87502,0.0,null
"""3""",0.87089,0.87089,0.0,null
"""4""",0.87229,0.87229,0.0,null
"""mean""",0.87291,0.87291,0.0,null


In [6]:
flips = int((np.load(cv.OOF_DIR / "exp_0005.npy").argmax(1) != oof_base.argmax(1)).sum())
moved = float(np.abs(np.load(cv.OOF_DIR / "exp_0005.npy") - oof_base).max())
print(f"probabilities moved by up to {moved:.3f}, argmax decisions flipped: {flips:,} of {train.height:,}")

probabilities moved by up to 0.016, argmax decisions flipped: 0 of 690,088


**H1 is a perfect zero — with a mechanism.** The indicator columns nudged the
probabilities (the trees *did* use them occasionally) but flipped essentially no
decisions: NaN routing already encodes everything the flags say. This is the cheapest
kind of negative — theory said "probably redundant", the experiment made it a fact.

**H2 missed.** The paired t is −0.3: the low-activity `fit` rows are apparently not
intense-but-brief exercisers; the ratio adds no separating direction:

In [7]:
cv.paired_diff("exp_0006", "exp_0001", train=train)

fold,exp_0006,exp_0001,diff,t
str,f64,f64,f64,f64
"""0""",0.8709,0.87121,-0.00031,null
"""1""",0.87457,0.87515,-0.00058,null
"""2""",0.87505,0.87502,0.00003,null
"""3""",0.87076,0.87089,-0.00013,null
"""4""",0.87298,0.87229,0.00069,null
"""mean""",0.87285,0.87291,-0.00006,-0.3


---
## 4. The resolution question — the idea the error loop could not see

Two negatives is a thin feature section, and the reason is worth more than the two
experiments. The error-profile loop asks **where** the model fails: which bin, which
segment, which class. It never asks **at what resolution** the model is allowed to see a
column — and that is where the remaining signal in this dataset turned out to live.

Do the arithmetic on what a tree can actually resolve. LightGBM bins every numeric
column into at most 255 bins before it ever looks for a split, then carves the space into
a few dozen leaves per tree. `sleep_duration` has 701 distinct values on a clean 0.01
grid, so ~3 distinct values share a bin, and the final model expresses the column through
a few dozen thresholds. **Any structure that lives at the resolution of single values is
invisible to that model at any capacity** — more trees do not help, because the
information was destroyed at binning time.

Is there such structure? The obvious test — scatter the per-value class rate and look
for jumps — cannot answer it, because three things are mixed in that scatter: binomial
noise (a value with 900 rows has SE ≈ 0.016), the column's own smooth trend, and genuine
value-level signal. `eda.exact_value_signal` separates them with a **split-half
replication test**: split the rows at random, compute per-value rates on each half,
subtract a leave-one-out local baseline from each (that removes the trend), then correlate
the two halves' residuals.

- noise does not replicate → correlation ≈ 0, SE ≈ `1/sqrt(n_values)`
- leftover trend replicates *and* is smooth → positive lag-1 autocorrelation
- exact-value signal replicates *and* is white → high correlation, lag-1 ≈ 0

In [8]:
signal = eda.exact_value_signal(train, io.NUMERIC_COLS, io.TARGET, classes=["unhealthy", "fit"])
signal.sort("true_sd", descending=True).select(
    "column", "class", "n_values", "base_rate", "resid_sd", "replication_r", "r_se",
    "lag1_autocorr", "true_sd", "verdict"
)

column,class,n_values,base_rate,resid_sd,replication_r,r_se,lag1_autocorr,true_sd,verdict
str,str,i64,f64,f64,f64,f64,f64,f64,str
"""sleep_duration""","""unhealthy""",607,0.0832,0.0972,0.935,0.041,-0.142,0.094,"""exact-value signal"""
"""water_intake""","""unhealthy""",319,0.0841,0.047,0.858,0.056,-0.169,0.0435,"""exact-value signal"""
"""sleep_duration""","""fit""",607,0.0582,0.0381,0.8,0.041,-0.101,0.0341,"""exact-value signal"""
"""heart_rate""","""unhealthy""",458,0.0846,0.0291,0.637,0.047,-0.069,0.0232,"""exact-value signal"""
"""water_intake""","""fit""",319,0.0579,0.0247,0.617,0.056,-0.085,0.0194,"""exact-value signal"""
"""heart_rate""","""fit""",458,0.0582,0.0216,0.328,0.047,0.028,0.0123,"""exact-value signal"""
"""bmi""","""fit""",1159,0.0571,0.0259,0.209,0.029,0.085,0.0118,"""exact-value signal"""
"""exercise_duration""","""fit""",654,0.0578,0.0206,0.135,0.039,-0.072,0.0076,"""noise"""
"""calorie_expenditure""","""fit""",1431,0.0581,0.0233,0.045,0.026,-0.044,0.005,"""noise"""


**Three of seven numeric columns carry exact-value signal, and it is not subtle.** For
`sleep_duration` / `unhealthy`: 607 frequent values, replication r = 0.94 against an SE
of 0.041 (≈23 SE), lag-1 −0.14 (white, as the leave-one-out baseline makes it), and a
replicating residual sd of **0.094** — larger than the class's own 8.4% base rate. The
same test reads r ≈ 0 for `step_count`, `bmi`, `exercise_duration` and
`calorie_expenditure`, which is what a working null looks like.

Look at the raw rows behind that verdict — neighbouring values 0.01 apart:

In [9]:
eda.value_target_rates(train, "sleep_duration", io.TARGET, classes=["unhealthy"]).slice(200, 10)

sleep_duration,n,unhealthy,se_unhealthy
f32,u32,f64,f64
5.54,908,0.440529,0.0165
5.55,1582,0.553097,0.0125
5.56,454,0.303965,0.0216
5.57,972,0.25823,0.014
5.58,978,0.164622,0.0119
5.59,1969,0.346369,0.0107
5.6,853,0.393904,0.0167
5.61,1210,0.571901,0.0142
5.62,972,0.463992,0.016


`5.55` hours → 55.3% unhealthy. `5.58` hours → 16.5%. Each rate carries an SE of about
0.012, so the two rates are 22 SE apart — a **decision-flipping swing between two values three
hundredths of an hour apart** — and then it swings back. No monotone transform, no ratio, and no number of
axis-aligned splits at 255-bin resolution can express that. A per-value encoding can.

### 4.1 Where the prize actually is — and where it is not

A verdict is not an effect size. Run the same test inside bands of `sleep_duration` and
compare each band's replicating residual to its base rate:

In [10]:
rows = []
for lo, hi in [(3.0, 5.5), (5.5, 6.2), (6.2, 7.0), (7.0, 10.1)]:
    band = train.filter(pl.col("sleep_duration").is_between(lo, hi, closed="left"))
    out = eda.exact_value_signal(band, ["sleep_duration"], io.TARGET, classes=["unhealthy"])
    rows.append(out.with_columns(pl.lit(f"[{lo}, {hi})").alias("band")))
pl.concat(rows).select("band", "n_values", "base_rate", "true_sd", "replication_r", "verdict")

band,n_values,base_rate,true_sd,replication_r,verdict
str,i64,f64,f64,f64,str
"""[3.0, 5.5)""",160,0.3945,0.173,0.935,"""exact-value signal"""
"""[5.5, 6.2)""",70,0.265,0.0961,0.97,"""exact-value signal"""
"""[6.2, 7.0)""",80,0.0013,0.0015,0.577,"""exact-value signal"""
"""[7.0, 10.1)""",297,0.0027,0.0021,0.288,"""exact-value signal"""


This is the honest reading, and it corrects the tempting one. The structure replicates in
**every** band — but in the mid-band (6.2–7.0) the `unhealthy` base rate is 0.0013 and
the replicating residual is 0.0015. Real, and worth nothing: there are almost no
`unhealthy` rows there to win. Section 1's mid-band crater is **not** rescued by this
feature; that segment stays irreducible.

Where the prize sits is the low band: base rate 0.394 with a replicating residual of
0.173, i.e. individual values swinging between roughly 10% and 70% unhealthy. That is the
region where `at-risk` and `unhealthy` are actually contested — exactly where a sharper
boundary converts into recall.

### 4.2 Would it survive to test time?

A per-value encoding is useless if test rows land on values train never saw often enough
to estimate:

In [11]:
eda.value_coverage(train, io.load_test(), io.NUMERIC_COLS)

column,frequent_values,test_rows_covered,pct_test_covered
str,i64,i64,f64
"""sleep_duration""",646,262869,88.9
"""heart_rate""",484,292218,98.8
"""bmi""",1289,288423,97.5
"""calorie_expenditure""",1703,271070,91.7
"""step_count""",6254,255955,86.5
"""exercise_duration""",706,292203,98.8
"""water_intake""",330,276792,93.6


89–99% coverage on the three columns that matter. The encoding would apply to nearly
every test row, with a global-mean fallback for the rest.

### 4.3 Why this is a licence to run one more experiment

Notebook 07's stopping rule declines any experiment whose outcome could not be read even
if it won. This one can be read, because it arrives with a mechanism and a prior effect
size rather than a hope:

- **Mechanism:** the playground generator resamples numeric values from a finite support,
  so a repeated value behaves like a high-cardinality *category*, not a point on a
  continuum. The test above is that mechanism's fingerprint, measured on our own data.
- **Prior effect size:** the 11th-place writeup reports 39 fold-fitted target-encoding
  features (13 columns × 3 classes) moving XGBoost 0.94890 → 0.94956, +0.0007
  (LEARNING.md). Our own private score sits 0.0011 below the winner's.
- **The trap that comes with it:** the same writeup screened this idea on 70k rows and
  read **−0.0017**, the opposite sign of its +0.0012 at full scale. Per-value statistics
  need the repeats to exist, and a 10% subsample destroys them. Screen this idea on full
  rows with fewer folds — never on fewer rows.

And the leak discipline is not optional here: a per-value target mean fitted on all rows
would let each row see its own label. The encoding has to be fitted **inside the fold**
(CLAUDE.md rule 3), which is why it belongs in `cv.py`'s fold loop and not in
`features.py`.

---
## 5. What the failures teach

Two negatives, one licence, and a loop worth carrying:

**profile errors → name the segment → one hypothesis per experiment → paired verdict →
log it either way.** Random feature ideas skip the first two steps, and that is why they
cost weeks.

But add the step this notebook was missing, because it is the one that found the real
feature: **ask what resolution the model can see, not only where it fails.** Binning
depth, leaf count, and cardinality are part of the feature question, not the tuning
question. The two ideas the error loop produced were both about *new columns*; the idea
that pays is about *the same column, encoded so the model can see it*.

The error profile itself is unchanged and still constrains everything: mid-band sleep and
missing sleep carry no minority signal at any resolution, so no feature will rescue them.
The remaining ~0.001 to the top of the leaderboard lives at the boundary in the bands
where the minorities actually are — and section 4 says how to reach it.